# A minimal OpenDP release

This notebook releases a single differentially private statistic with plain
[OpenDP](https://docs.opendp.org/) — no `noisyvalue` involved. It's the
baseline for [`opendp_noisyvalue.ipynb`](opendp_noisyvalue.ipynb), which
performs the *same* release through `noisyvalue.opendp` and gets back an
object that knows its own uncertainty instead of a bare number.

The task: a volunteer coordinator logs each volunteer's hours for the month
and wants to publish the total without revealing any individual's hours.

In [1]:
import numpy as np
from scipy import stats

import opendp.prelude as dp

dp.enable_features("contrib", "honest-but-curious")

## The data

Twenty volunteers, each contributing between 0 and 10 hours. The bound is a
public fact about the program (the sign-up form caps hours at 10) — it isn't
sensitive, so we're allowed to hard-code it into the query.

In [2]:
hours = [2.5, 6.0, 0.0, 10.0, 4.5, 7.5, 3.0, 8.0, 1.0, 9.5,
        5.0, 6.5, 2.0, 10.0, 0.5, 7.0, 3.5, 9.0, 4.0, 8.5]
true_sum = sum(hours)
true_sum

108.0

## Build the release

`make_sum` turns the list into a bounded sum; chaining it with
`make_gaussian` turns that sum into a measurement — a function that also
knows its own privacy cost. This mirrors `test_chains_with_transformation`
in `test/test_opendp.py`, which checks that `noisyvalue.opendp`'s wrapper
preserves exactly this chaining behavior.

In [3]:
trans = dp.t.make_sum(
    dp.vector_domain(dp.atom_domain(bounds=(0.0, 10.0), nan=False)),
    dp.symmetric_distance())

scale = 10.0
meas = trans >> dp.m.make_gaussian(trans.output_domain, trans.output_metric, scale)

# zero-concentrated-DP privacy loss (rho) for one volunteer joining or leaving
print(f"privacy loss (rho): {meas.map(1):.4f}")

privacy loss (rho): 0.5000


In [4]:
release = meas(hours)
print(f"true total:     {true_sum}")
print(f"released total: {release:.2f}")

true total:     108.0
released total: 106.42


## The release is just a float

`meas(hours)` hands back a plain number. Nothing about it says how it was
produced or how far it might be from the truth — that context lives only in
`scale`, which you have to keep track of separately and remember to apply
correctly to anything you compute from the release.

Running the mechanism again confirms the noise is real and varies draw to
draw:

In [5]:
repeats = np.array([meas(hours) for _ in range(2000)])
print(f"empirical mean:  {repeats.mean():.2f}  (true total: {true_sum})")
print(f"empirical std:   {repeats.std():.2f}  (mechanism scale: {scale})")

empirical mean:  108.02  (true total: 108.0)
empirical std:   10.21  (mechanism scale: 10.0)


To turn `release` into an interval you trust, you build it by hand from
the mechanism's known noise law — nothing does this for you, and if you lose
track of `scale` (or of which mechanism produced the number) the release
becomes unusable:

In [6]:
lo, hi = stats.norm.interval(0.95, loc=release, scale=scale)
print(f"released total: {release:.2f}")
print(f"95% interval:    [{lo:.2f}, {hi:.2f}]")

released total: 106.42
95% interval:    [86.82, 126.02]


## Further computation loses even that

Suppose we want average hours per volunteer, not the total. The count of
volunteers (20) is public, so this is just a division — but `release` is a
bare float, so the uncertainty has to be propagated by hand, again:

In [7]:
avg_release = release / len(hours)
avg_lo, avg_hi = lo / len(hours), hi / len(hours)
print(f"released average: {avg_release:.3f}")
print(f"95% interval:      [{avg_lo:.3f}, {avg_hi:.3f}]")

released average: 5.321
95% interval:      [4.341, 6.301]


Every downstream computation repeats this by-hand bookkeeping, and it's
easy to get wrong — dividing the interval's endpoints is only valid because
division by a constant is monotonic; anything less trivial (a sum of two
independent releases, a ratio of two releases) needs its own derivation.

See [`opendp_noisyvalue.ipynb`](opendp_noisyvalue.ipynb) for the same release
made through `noisyvalue.opendp`, where the posterior travels with the value
through exactly this kind of arithmetic.